### Silver Layer - Taxi Zone Lookup Silver Layer

This notebook updates the taxi_zone_lookup table with the new data using slowly changing dimensions(SCD) type 2.

In [0]:
# import statments
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import IntegerType, TimestampType


In [0]:
# loading the file
df = spark.read.format("csv").option("header", True).load("/Volumes/nyctaxi/00_landing/data_sources/taxi_zone_lookup")

In [0]:
# creating the dataframe with columns effective_date and end_date for SCD type 2
source_df = df.select(
    col("LocationID").cast(IntegerType()).alias("location_id"),
    col("Borough").alias("borough"),
    col("Zone").alias("zone"),
    col("service_zone"),
    current_timestamp().alias("effective_date"),
    lit(None).cast(TimestampType()).alias("end_date")
)

In [0]:
delta_table = DeltaTable.forName(spark, "nyctaxi.02_silver.taxi_zone_lookup")


In [0]:
target_df = delta_table.toDF()

In [0]:
# ============================================================
# Step 1: Find changed active records
# ============================================================

changed_rows = (
    source_df.alias("source")
    .join(
        target_df.alias("target"),
        (col("source.location_id") == col("target.location_id"))
        & col("target.end_date").isNull(),
        "inner"
    )
    .filter(
        (col("source.borough") != col("target.borough"))
        | (col("source.zone") != col("target.zone"))
        | (col("source.service_zone") != col("target.service_zone"))
    )
    .select("source.*")
)

In [0]:
# ============================================================
# Step 2: Stage rows for MERGE
#
# Existing rows are matched and expired.
# New rows are inserted.
# Changed rows appear twice:
#   1. NULL merge key -> INSERT
#   2. location_id    -> UPDATE existing row
# ============================================================

staged_updates = (
    changed_rows
    .withColumn("merge_key", lit(None).cast(IntegerType()))
    .unionByName(
        changed_rows.withColumn("merge_key", col("location_id"))
    )
    .unionByName(
        source_df.alias("source")
        .join(
            target_df.alias("target"),
            (col("source.location_id") == col("target.location_id"))
            & col("target.end_date").isNull(),
            "left_anti"
        )
        .withColumn("merge_key", col("location_id"))
    )
)


In [0]:

# ============================================================
# Step 3: Execute SCD Type 2 MERGE
# ============================================================

(
    delta_table.alias("target")
    .merge(
        staged_updates.alias("source"),
        """
        target.location_id = source.merge_key
        AND target.end_date IS NULL
        """
    )
    .whenMatchedUpdate(
        set={
            "end_date": "current_timestamp()"
        }
    )
    .whenNotMatchedInsert(
        values={
            "location_id": "source.location_id",
            "borough": "source.borough",
            "zone": "source.zone",
            "service_zone": "source.service_zone",
            "effective_date": "source.effective_date",
            "end_date": "source.end_date"
        }
    )
    .execute()
)